In [ ]:
from bs4 import BeautifulSoup
import requests

response = requests.get(r"https://www.consumer.vic.gov.au/housing/renting/starting-and-changing-rental-agreements/resources-and-guides-for-renters/renters-guide/finding-and-applying-for-a-rental-property")


In [ ]:
html = response.content.decode("utf-8")

html

In [ ]:
with open(r"..\\raw_data\\Finding_and_applying_for_a_rental_property.txt", "w") as f:
    f.write(html)

In [ ]:
with open(r"..\\raw_data\\Introduction_and_helpful_contacts.txt", "r") as f:
    html = f.read()

In [ ]:
from langchain_text_splitters import HTMLHeaderTextSplitter

headers_to_split_on = [("h1", "h1"), ("h3", "h3")]


html_header_splitter = HTMLHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    return_each_element=False,
)

header_chunks = html_header_splitter.split_text(html)

In [ ]:
header_chunks

In [ ]:
from langchain_text_splitters import HTMLSemanticPreservingSplitter

def table_to_text(table):
    headers = [th.get_text(" ", strip=True) for th in table.find_all("th")]
    lines = []
    for tr in table.find_all("tr"):
        cells = [td.get_text(" ", strip=True).replace("\xa0", " ")
                 for td in tr.find_all("td")]
        if cells:
            lines.append("; ".join(f"{h}: {c}" for h, c in zip(headers, cells)))
    return "\n".join(lines)

html_semantic_splitter = HTMLSemanticPreservingSplitter(
    headers_to_split_on=headers_to_split_on,
    max_chunk_size=1500,
    chunk_overlap=0,
    separators=["\n\n", "\n", ".", " "],
    elements_to_preserve=["table", "ul", "ol"],
    custom_handlers={"table": table_to_text},
    external_metadata= {"url": "the_url"}, # We add the url as we will implement RAG with Sources
)

semantic_chunks = html_semantic_splitter.split_text(html)

In [ ]:
semantic_chunks

We can see that it is better to use the HTMLSemanticPreservingSplitter because of its 'custom_handlers' attribute that allows us to reformat table data and possibly other types of data in a more semantically meaningful way.

I also believe that HTMLSemanticPreservingSplitter will perform better than true Semantic Chunking because the structure of the web pages are very well-defined.

One big issue with HTMLSemanticPreservingSplitter is that each chunk has only its parent heading in its metadata, not all its ancestor heading. This can make our chunks lose meaningful context. In such case, the type of header metadata for headers provided by HTMLHeaderTextSplitter is highly preferred.

Direct metadata filtering on the headers is not reasonable as the header text is not exactly unique (meaning they could be written in multiple ways by the user, unlike product codes).

The best way to handle this is to preprocess the chunks so that the header tags are present as in HTMLHeaderTextSplitter and append all ancestor headers to the content as a prefix.

In [ ]:
import re

HEADER_KEY = re.compile(r"^h[1-6]$")

def post_process_header_tags(semantic_chunks):
    # List to contain header texts of h1-h6.
    headers = []
    # Variable to hold current smallest header traversed
    curr_smallest_header_traversed = 0
    for chunk in semantic_chunks:
        if chunk.metadata:
            for key, value in chunk.metadata.items():
                if HEADER_KEY.match(key):
                    header_index = int(key[-1])

                    # Adjust the header hierarchy for the chunk
                    if header_index == curr_smallest_header_traversed:
                        headers.pop()
                        headers.append((header_index,value))
                    elif header_index > curr_smallest_header_traversed:
                        curr_smallest_header_traversed = header_index
                        headers.append((header_index, value))
                    else:
                        curr_smallest_header_traversed = header_index
                        for i in range(1, len(headers) + 1):
                            if headers[(-1*i)][0] >= header_index:
                                headers.pop()
                            else:
                                break

                        headers.append((header_index, value))

                    chunk_header = ""

                    # Create the chunk header string
                    for header in headers:
                        chunk_header = chunk_header + header[1] + " - "

                    # Append all ancestor headers to the chunk content
                    chunk.page_content = chunk_header + chunk.page_content

In [ ]:
post_process_header_tags(semantic_chunks)


In [ ]:
semantic_chunks